<a href="https://colab.research.google.com/github/guitorte/audio/blob/ccr-8be874fb-rdbsnf/neural-upscaler/FlashSR_Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ FlashSR trial

**Experimental.** A quick test of [FlashSR](https://github.com/jakeoneijk/FlashSR_Inference), a one-step model that does the same job as AudioSR (step ③ of the Neural Audio Upscaler): rebuilding the highs an MP3 cut off. Its authors report similar quality at about 22 times the speed.

This notebook restores **a short excerpt of one song** and answers three questions:

1. Does FlashSR load and run in Colab?
2. How fast is it (seconds of work per minute of music, and GPU memory)?
3. How do its highs sound, next to the original and, if you tick the box, next to AudioSR?

It uses the same splice as the main notebook: only the band **above** the detected cutoff comes from the model, and everything below it stays the original audio. Your songs are never changed; results go to `upscaled/flashsr_trial/` inside your folder.

**How to use:** `Runtime ▸ Change runtime type ▸ T4 GPU`, then run the 3 cells in order. The first run downloads about 3.3 GB of weights.

*FlashSR's repository states no licence, so treat the results as a personal experiment.*

In [ ]:
#@title 1 · Connect Google Drive and list your songs
DRIVE_FOLDER = "áudio"  #@param {type:"string"}
#@markdown Path inside *My Drive* (e.g. `áudio` or `Music/áudio`). Accents and capitals don't need to match exactly.

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "ccr-8be874fb-rdbsnf"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/neural-upscaler")
import upscaler as U

FOLDER = U.find_folder(DRIVE_FOLDER)
FILES = U.list_audio(FOLDER)
print(f"📁 {FOLDER}  ({len(FILES)} songs)\n")
for i, f in enumerate(FILES, 1):
    print(f"{i:3d}. {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    gpu = None
print(f"\nGPU: {gpu or '⚠️ none. Use Runtime ▸ Change runtime type ▸ T4 GPU'}")

In [ ]:
#@title 2 · Try FlashSR on a short excerpt of one song
SONG = "1"  #@param {type:"string"}
#@markdown One song: its number from the list, or part of its name. Pick an MP3 that sounds dull: cell 2 prints where its highs stop, and 16 kHz or lower makes a fair test.
START_AT_SECONDS = 45  #@param {type:"number"}
CLIP_SECONDS = 20  #@param {type:"number"}
CUTOFF_HZ = 0  #@param {type:"integer"}
#@markdown `0` = detect the lossy shelf automatically. Set e.g. `16000` to force a rebuild above that frequency.
COMPARE_WITH_AUDIOSR = False  #@param {type:"boolean"}
#@markdown Also run AudioSR on the same excerpt, to compare speed and sound. ⏱ Adds several minutes: AudioSR sets up its own environment and downloads its own weights. It runs first; if it fails, FlashSR still runs.

import flashsr_trial as T
song = U.select_files(FILES, SONG)[0]
RES = T.trial(song, FOLDER, START_AT_SECONDS, CLIP_SECONDS, CUTOFF_HZ, COMPARE_WITH_AUDIOSR)

In [ ]:
#@title 3 · Listen and look (level-matched, so louder doesn't fool your ears)
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display

CLIPS = T.clips(RES)
for label, audio in CLIPS.items():
    display(Markdown(f"**{label}**")); display(Audio(np.clip(audio.T, -1, 1), rate=T.SR, normalize=False))
fig, axes = plt.subplots(1, len(CLIPS), figsize=(7 * len(CLIPS), 3.6), sharey=True, squeeze=False)
for ax, (label, audio) in zip(axes[0], CLIPS.items()):
    ax.specgram(audio.mean(axis=1), NFFT=2048, Fs=T.SR, noverlap=1024, cmap="magma", vmin=-130, vmax=-20)
    ax.axhline(RES["cutoff_hz"], color="white", lw=0.6, ls="--")
    ax.set_title(label); ax.set_xlabel("seconds"); ax.set_ylim(0, T.SR / 2)
axes[0][0].set_ylabel("Hz"); plt.tight_layout(); plt.show()

### What to look for
- **Pick the right song.** FlashSR was trained on audio cut at 16 kHz or lower. If cell 2 says the song already reaches higher than that, the result says little; try a duller MP3.
- **The dashed line is the cutoff.** Below it the three pictures should look identical; above it, "Before" is empty and the others are filled in.
- **Speed:** the summary under cell 2 gives seconds of work per minute of music. for AudioSR the main notebook quotes 10+ minutes per song on a T4.
- **Sound:** listen for hiss, a metallic sheen, or highs that pump in 5-second blocks.
- The numbers are also saved as `<song>_trial.json` next to the WAV files.
- If cell 2 stops with an error, the lines printed **above** the red traceback explain it.